# Classification Track (Part A) — Adult Census Income Prediction
### 23CSE301 Machine Learning Capstone — Review 1

**Dataset:** UCI Adult / Census Income dataset (`adult.data` + `adult.test` merged) — 48,842 records of U.S. census respondents.

**Goal:** Predict whether a person's annual income exceeds $50K (`income`: `<=50K` / `>50K`) from demographic and employment features.

**Scope note:** Per the course guidelines, Classification Part A (Review 1) covers the first 5 algorithms — Logistic Regression, KNN,
Naive Bayes, Decision Tree, and SVM. Part B (Random Forest, AdaBoost, Gradient Boosting, Bagging, MLP) and the consolidated 10-algorithm
table are Review 2 scope and are not included here.

All random operations use `random_state=42` for reproducibility.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                              roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve)

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 100

## Section A — Dataset Loading & Exploratory Data Analysis

### A1. Dataset Loading & Audit

**Note on missing values:** the raw UCI Adult files mark missing entries with the literal string `?` rather than leaving the
field blank. `na_values=['?']` tells `pandas` to parse those cells as genuine `NaN` on load, so `.isna().sum()` below correctly reports
them as missing instead of silently treating `"?"` as a valid category value.

In [2]:
df = pd.read_csv('../data/adult.csv', na_values=['?'])
print("Shape:", df.shape)
df.head()

In [3]:
print("Data types:\n")
print(df.dtypes)
print("\nMissing values per column:\n")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [4]:
print("Target ('income') class distribution:\n")
print(df['income'].value_counts())
print()
print(df['income'].value_counts(normalize=True).round(4))

**Observation:** 48,842 rows, 15 columns. Three categorical columns (`workclass`, `occupation`, `native-country`) contain missing
values (originally encoded as `?` in the raw files) — around 5–6% of rows are affected. There are 52 exact duplicate rows. The target is
**imbalanced**: ~76% earn `<=50K` vs. ~24% earn `>50K`, which is why we use a **stratified** train/test split and report **weighted** F1/
Precision/Recall (accuracy alone would be misleading on an imbalanced target).

### A2. EDA Visualisations

In [5]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='income', hue='income', palette='colorblind', legend=False)
plt.title('Target Distribution — Income Class')
plt.xlabel('Income'); plt.ylabel('Count')
plt.tight_layout()
plt.savefig('clf_target_distribution.png', bbox_inches='tight')
plt.show()